<!-- NB_DOC_INTRO_V1 -->
# Streamlit — briques UI pour data apps

> 📚 **Doc thematique** : [docs/07_BDD_DE.md](docs/07_BDD_DE.md) (Bases de donnees / Data Engineering / Web)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Streamlit** = framework Python pour creer des **web apps de DS** en quelques lignes (sans HTML/CSS/JS). Standard pour : dashboards, demos ML, prototypes interactifs, internal tools.

Note : Streamlit doit etre lance en CLI (`streamlit run app.py`), pas dans un notebook. Ce notebook execute donc du **code de reference** + tests Python qu'on peut valider sans serveur.

## Plan

1. Concepts (st.write, layouts, state)
2. Widgets (text_input, slider, button, file_uploader)
3. DataFrames + charts
4. Caching (@st.cache_data, @st.cache_resource)
5. Session state
6. Multi-page apps
7. Layouts (columns, tabs, expander, sidebar)
8. Chat UI (st.chat_message, st.chat_input)
9. Deploiement (Streamlit Cloud, HF Spaces, Docker)
10. Pieges + References


In [ ]:
import warnings
warnings.filterwarnings("ignore")
print("Streamlit s'execute en CLI : 'streamlit run app.py'")
print("Ce notebook contient le code de reference + valide la syntaxe.")

## 1. Hello world — app.py minimal

```python
streamlit run app.py
```

In [ ]:
hello_app = '''
# app.py
import streamlit as st

st.title("Mon app Streamlit")
st.write("Hello, World !")

name = st.text_input("Ton prenom :")
if name:
    st.write(f"Bonjour {name} !")
'''
print(hello_app)

## 2. Widgets — interagir avec l'utilisateur

In [ ]:
widgets_example = '''
import streamlit as st

# Text input
name = st.text_input("Nom", value="Alice")

# Slider
age = st.slider("Age", min_value=0, max_value=100, value=30)

# Number input
score = st.number_input("Score", min_value=0.0, max_value=100.0, value=50.0, step=0.5)

# Selectbox (dropdown)
category = st.selectbox("Category", ["A", "B", "C"])

# Multiselect
tags = st.multiselect("Tags", ["ml", "dl", "nlp", "ts"])

# Checkbox
active = st.checkbox("Actif")

# Radio
gender = st.radio("Genre", ["M", "F", "Autre"])

# Date / time
birth = st.date_input("Date de naissance")

# Color picker
color = st.color_picker("Couleur preferee", "#00f900")

# Buttons
if st.button("Valider"):
    st.success(f"Merci {name} !")

# File upload
uploaded = st.file_uploader("CSV", type=["csv"])
if uploaded:
    import pandas as pd
    df = pd.read_csv(uploaded)
    st.dataframe(df)
'''
print(widgets_example)

## 3. DataFrames + charts natifs

In [ ]:
df_and_charts = '''
import streamlit as st
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "month": pd.date_range("2024-01-01", periods=12, freq="MS"),
    "sales": np.random.randint(100, 200, 12),
    "category": np.random.choice(["A", "B"], 12),
})

# Dataframe interactif (sortable, filtrable)
st.dataframe(df, use_container_width=True)

# Table statique
st.table(df.head())

# Charts natifs
st.line_chart(df.set_index("month")["sales"])
st.bar_chart(df.set_index("month")["sales"])
st.area_chart(df.set_index("month")["sales"])

# Custom plot (matplotlib / plotly / altair)
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot(df["month"], df["sales"], marker="o")
st.pyplot(fig)

# Plotly (interactif)
import plotly.express as px
fig = px.scatter(df, x="month", y="sales", color="category")
st.plotly_chart(fig, use_container_width=True)
'''
print(df_and_charts)

## 4. Caching — `@st.cache_data` et `@st.cache_resource`

In [ ]:
caching_example = '''
import streamlit as st
import pandas as pd

# @st.cache_data : pour donnees (DataFrame, dict, list, ...)
@st.cache_data
def load_data(path):
    return pd.read_csv(path)  # ne s'execute qu'une fois par valeur unique de `path`

# @st.cache_resource : pour ressources non-serializables (modeles ML, connexions DB)
@st.cache_resource
def load_model():
    import joblib
    return joblib.load("model.pkl")    # charge UNE FOIS, partage entre sessions

df = load_data("data.csv")
model = load_model()
pred = model.predict(df.values)
'''
print(caching_example)
print("\nDifference :")
print("  @cache_data    : copie a chaque get (safe, evite mutations)")
print("  @cache_resource: refere partagee (pour modeles lourds)")

## 5. Session state — persister entre interactions

In [ ]:
session_state = '''
import streamlit as st

# Initialize once
if "counter" not in st.session_state:
    st.session_state.counter = 0

if st.button("Increment"):
    st.session_state.counter += 1

if st.button("Reset"):
    st.session_state.counter = 0

st.write(f"Counter : {st.session_state.counter}")

# Pour formulaires multi-step : stocker chaque etape
if "step" not in st.session_state:
    st.session_state.step = 1

if st.session_state.step == 1:
    name = st.text_input("Nom")
    if name and st.button("Next"):
        st.session_state.name = name
        st.session_state.step = 2
        st.rerun()

if st.session_state.step == 2:
    st.write(f"Hello {st.session_state.name}!")
'''
print(session_state)

## 6. Layouts — columns, tabs, expander, sidebar

In [ ]:
layouts_example = '''
import streamlit as st

# Sidebar (toujours visible a gauche)
with st.sidebar:
    st.title("Filtres")
    region = st.selectbox("Region", ["N", "S", "E", "W"])
    period = st.slider("Periode", 1, 12, 6)

# Columns
col1, col2, col3 = st.columns(3)
with col1:
    st.metric("CA", "100k€", "+5%")
with col2:
    st.metric("Clients", "1.2k", "+12%")
with col3:
    st.metric("Conv", "3.4%", "-0.2%")

# Tabs
tab1, tab2, tab3 = st.tabs(["Overview", "Analytics", "Settings"])
with tab1:
    st.write("Overview content...")
with tab2:
    st.write("Analytics content...")
with tab3:
    st.write("Settings content...")

# Expander (collapsible)
with st.expander("Details"):
    st.write("Detailed info hidden by default.")
    st.code("python code...", language="python")
'''
print(layouts_example)

## 7. Chat UI — `st.chat_message`, `st.chat_input`

In [ ]:
chat_example = '''
import streamlit as st

st.title("Chatbot demo")

# Init history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Render history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# User input
if prompt := st.chat_input("Votre message"):
    # Add user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # LLM response (placeholder)
    response = f"Echo : {prompt}"  # remplacer par appel OpenAI/Ollama
    st.session_state.messages.append({"role": "assistant", "content": response})
    with st.chat_message("assistant"):
        st.markdown(response)
'''
print(chat_example)

## 8. Multi-page apps

```
project/
├── Home.py            # page principale
└── pages/
    ├── 1_📊_Dashboard.py
    ├── 2_📈_Forecast.py
    └── 3_⚙️_Settings.py
```

Streamlit detecte automatiquement le dossier `pages/` et cree un menu lateral.

## 9. Deploiement

| Plateforme | Cout | Notes |
|---|---|---|
| **Streamlit Cloud** | Gratuit (limite) | Push GitHub → deploy auto |
| **HuggingFace Spaces** | Gratuit (16 GB RAM) | Ideal pour app + modele ML |
| **Render** | Free tier | docker / native |
| **Heroku** | Payant | Container ready |
| **GCP Cloud Run / AWS App Runner** | Payant | Production scalable |
| **Docker self-host** | Server cost | Total control |

```dockerfile
# Dockerfile minimal
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]
```

## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Charger modele a chaque rerun | `@st.cache_resource` |
| Pas utiliser cache_data sur read_csv | Recharge a chaque interaction |
| Setter session_state sans guard | `if 'x' not in st.session_state` |
| Pas catch les exceptions dans buttons | `try/except` + `st.error(...)` |
| Logique metier dans le main script | Externaliser dans modules Python |
| Secrets en clair (`st.secrets`) | TOML file (gitignored) |
| Pas de `st.spinner` sur ops longues | Mauvaise UX |
| Tout dans une page enorme | Multi-page apps |
| Pas tester localement avant deploy | `streamlit run app.py` |


## References

### Documentation
- Streamlit docs : https://docs.streamlit.io/
- Streamlit gallery : https://streamlit.io/gallery
- Streamlit Cloud : https://share.streamlit.io/
- HF Spaces (Streamlit) : https://huggingface.co/docs/hub/spaces-sdks-streamlit

### Voir aussi
- [Flask_API.ipynb](Flask_API.ipynb)
- [FastAPI_API.ipynb](FastAPI_API.ipynb)
- [AI_Prompt_Engineering.ipynb](AI_Prompt_Engineering.ipynb)
